# Домашнее задание № 3

## 1. Корень числа
Бинарным поиском ищем наибольшее `k`, для которого `k*k <= n`. Границы — от 0 до `n`. На каждом шаге отбрасываем половину, поэтому **O(log n)** по времени, **O(1)** по памяти.

In [1]:
def integer_sqrt(n):
    # Для целого n >= 0 ищем наибольшее целое k, такое что k * k <= n.
    # Все кандидаты лежат в [0, n]; ans хранит лучший уже найденный ответ.
    left, right = 0, n
    ans = 0

    while left <= right:
        mid = (left + right) // 2  # Проверяем середину оставшегося диапазона.
        if mid * mid <= n:
            # mid подходит, но справа может быть ещё больший подходящий кандидат.
            ans = mid
            left = mid + 1
        else:
            # Если mid слишком велик, все большие числа тоже не подходят.
            right = mid - 1

    # Каждая итерация убирает примерно половину кандидатов.
    # Например, для 10 ответ 3: 3² <= 10, а 4² уже больше 10.
    return ans
assert integer_sqrt(0) == 0
assert integer_sqrt(10) == 3
assert integer_sqrt(25) == 5
print(integer_sqrt(26))

5


## 2. Два ксерокса
Сначала один ксерокс делает первую копию за `min(x, y)`. Дальше оба работают вместе. За `t` минут получится `t//x + t//y` копий. Бинарным поиском находим минимальное подходящее время. **O(log(n·min(x,y)))**, память **O(1)**.

In [2]:
def min_copy_time(n, x, y):
    # n — сколько новых копий нужно; x и y — положительное время одной копии на каждом аппарате.
    if n == 0:
        return 0

    # Сначала есть только оригинал, поэтому одновременно работать может один аппарат.
    # Первую копию выгодно сделать быстрым аппаратом. После неё лист есть у каждого.
    first = min(x, y)
    # Ищем ДОПОЛНИТЕЛЬНОЕ время на n - 1 копий после первой.
    # Правая граница точно достаточна: быстрый аппарат справится за неё даже один.
    left, right = 0, (n - 1) * first

    while left < right:
        mid = (left + right) // 2
        # За mid единиц времени аппараты успеют сделать mid // x и mid // y полных копий.
        # Если времени достаточно, любое большее время тоже подходит: возможен бинарный поиск.
        if mid // x + mid // y >= n - 1:
            right = mid  # mid подходит и сам может быть минимумом, поэтому сохраняем его.
        else:
            left = mid + 1  # mid не подходит, как и всё меньшее время.

    # Границы сошлись на минимальном достаточном времени после первой копии.
    # При n = 1 обе границы сразу равны 0, остаётся только время первой копии.
    return first + left
assert min_copy_time(1, 1, 2) == 1
assert min_copy_time(4, 1, 2) == 3
assert min_copy_time(5, 1, 1) == 3
print(min_copy_time(4, 1, 2))

3


## 3. Накормить животных
Сортируем животных и порции. Каждую порцию отдаём самому «непривередливому» животному, которому её хватает. Так большие порции остаются для остальных. **O(a log a + p log p)**, где `a` — животные, `p` — порции.

In [3]:
def max_fed_animals(animals, portions):
    # animals хранит минимальные размеры порций, нужные животным.
    # Сортируем потребности и порции от маленьких к большим.
    animals = sorted(animals)
    portions = sorted(portions)
    i = 0  # Сколько животных накормлено; одновременно индекс следующего голодного.

    for food in portions:
        # Если порции мало даже для наименее требовательного животного, пропускаем её:
        # остальные требуют ещё больше. Каждую порцию рассматриваем ровно один раз.
        if i < len(animals) and food >= animals[i]:
            # Отдаём подходящую порцию этому животному и переходим к следующему.
            # Большие порции сохраняются для более требовательных животных.
            i += 1

    return i  # Каждый сдвиг i соответствует ровно одному накормленному животному.
assert max_fed_animals([2, 3, 4], [1, 2, 3]) == 2
assert max_fed_animals([1, 2, 3], [3]) == 1
print(max_fed_animals([2, 3, 4], [1, 2, 3]))

2


## 4. Разница между строками
Применяем XOR ко всем символам обеих строк. Одинаковые символы сокращаются, остаётся добавленный. **O(n)** по времени, **O(1)** по памяти.

In [4]:
def find_added_character(a, b):
    # Условие: b содержит все символы a (возможно, в другом порядке) и один дополнительный.
    # XOR (^) работает с битами чисел: x ^ x = 0, а 0 ^ x = x.
    # Порядок применения XOR не важен, поэтому одинаковые коды сократятся попарно.
    result = 0
    for c in a + b:
        result ^= ord(c)  # ord переводит символ в числовой код; ^= накапливает XOR.
    # Все общие символы исчезли, остался код добавленного. Повторяющиеся буквы тоже работают.
    # Например, для 'ab' и 'baa': a ^ b ^ b ^ a ^ a = a (имеются в виду коды букв).
    return chr(result)  # Обратно переводим числовой код в символ.
assert find_added_character('abcd', 'cadbe') == 'e'
assert find_added_character('aabb', 'ababc') == 'c'
print(find_added_character('abcd', 'cadbe'))

e


## 5. Сумма двух элементов
Храним уже пройденные числа в словаре. Для каждого ищем `target - x`. Так не нужен второй цикл. **O(n)** в среднем по времени и **O(n)** по памяти.

In [5]:
def two_sum(nums, target):
    # Словарь хранит уже встреченные числа как ключи, их индексы — как значения.
    # Для ответа здесь нужны сами числа; индексы в словаре напрямую не используются.
    seen = {}

    for i, x in enumerate(nums):
        # Чтобы x + другое_число = target, второе число должно равняться target - x.
        # Проверяем его наличие среди УЖЕ просмотренных элементов за O(1) в среднем.
        if target - x in seen:
            return [target - x, x]
        # Добавляем x только после проверки, чтобы не использовать один элемент дважды.
        # Поэтому [3] не даст пару для 6, а [3, 3] даст её на втором элементе.
        seen[x] = i

    return []  # Просмотрели все элементы и ни разу не нашли дополнение.
assert two_sum([2, 7, 11, 15], 9) == [2, 7]
assert two_sum([3, 3], 6) == [3, 3]
assert two_sum([1, 2], 10) == []
print(two_sum([2, 7, 11, 15], 9))

[2, 7]


## 6. Массив анаграмм
Считаем частоты букв в каждом слове. Одинаковые наборы частот — одна группа в словаре. **O(S)** в среднем по времени и памяти, где `S` — суммарная длина строк.

In [6]:
def group_anagrams(strs):
    # Анаграммы содержат одинаковые буквы с одинаковыми частотами, порядок букв не важен.
    # groups связывает набор частот с группой подходящих слов.
    groups = {}

    for word in strs:
        count = {}  # Для каждого слова заново считаем, сколько раз встретилась каждая буква.
        for c in word:
            count[c] = count.get(c, 0) + 1  # Если буквы ещё нет в словаре, начинаем с нуля.

        # count.items() содержит пары (буква, частота). Обычный словарь нельзя сделать ключом.
        # frozenset — неизменяемое множество: подходит как ключ и не учитывает порядок пар.
        # Поэтому 'eat' и 'tea' получат одинаковый ключ, а 'tee' — другой.
        key = frozenset(count.items())
        # setdefault возвращает список для ключа, а при его отсутствии сначала создаёт [].
        groups.setdefault(key, []).append(word)

    return list(groups.values())  # Нужны только группы слов, вспомогательные ключи отбрасываем.
result = group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat'])
assert sorted(map(sorted, result)) == sorted(map(sorted, [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]))
print(result)

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
